# Latent-class fractional land-use function: first pass

このノートブックは、既存の `cropland_fraction_two_stage_lgbm.ipynb` を置き換えるものではなく、
第二段階（農地があるセルにおける cropland fraction）の構造を検証するための追加分析です。

目的は、世界全体で一つの関数を仮定するのではなく、相ごとに異なる明示的な関数

```text
logit(E[cropland fraction | positive, X, regime=k])
    = alpha_k + X beta_k
```

を推定することです。`regime` は事前に地理で決めず、有限混合モデルで潜在的に推定します。

注意：この第一版は、fractional-logitのquasi-likelihoodを使った有限混合モデルです。
したがって、まずは相別の反応構造が予測・解釈の両面で安定するかを確認します。

In [ ]:
from __future__ import annotations

import gc
import warnings
from pathlib import Path

import netCDF4
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path(r'C:\masterresearch\Comparative_advantage')
GAEZ_DIR = ROOT / 'GAEZ'
HYDE_DIR = ROOT / 'HYDE3.4'
CROPLAND_DIR = HYDE_DIR / 'cropland_npys'
DIST_DIR = ROOT / 'distance_to_cities'
GLOFAS_DIR = ROOT / 'GloFAS' / 'processed_5min'
FEATURE_CACHE = GAEZ_DIR / 'CroplandRegression' / 'features_cache'
OUT_DIR = GAEZ_DIR / 'CroplandRegression' / 'latent_class_fractional_first_pass'
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2024
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
N_SPLITS = 5
K_VALUES = [1, 2, 3, 4]
MAX_MIXTURE_TRAIN_ROWS = 15_000
EM_MAX_ITER = 25
EM_TOL = 1e-4
L2_ALPHA = 1e-4
EPS = 1e-6
RANDOM_SEED = 42

print('Output directory:', OUT_DIR)

## 1. 既存ノートブックと同じ特徴量を再構成する

元ノートブックを一度実行して特徴量キャッシュを作成してから、このノートブックを実行してください。
ここでは、現在の特徴量をそのまま使い、モデルの構造だけを変更します。

In [ ]:
def clean_float(array, nodata_values=(-9999, 65535)):
    out = np.asarray(array, dtype=np.float32).copy()
    for value in nodata_values:
        out[out == value] = np.nan
    out[~np.isfinite(out)] = np.nan
    return out


def safe_log1p(array):
    out = np.asarray(array, dtype=np.float32)
    out = np.where(np.isfinite(out) & (out > 0), out, 0.0)
    return np.log1p(out).astype(np.float32)


def require(path):
    if not path.exists():
        raise FileNotFoundError(
            f'Missing cached input: {path}\n'
            'Run the preprocessing cells in cropland_fraction_two_stage_lgbm.ipynb first.'
        )
    return path


lat = np.load(require(CROPLAND_DIR / 'lat.npy'))
lon = np.load(require(CROPLAND_DIR / 'lon.npy'))
cropland_all = np.load(require(CROPLAND_DIR / 'cropland_fraction_1950_2024.npy'), mmap_mode='r')
years = np.load(require(CROPLAND_DIR / 'years.npy'))
year_idx = int(np.where(years == YEAR)[0][0])
cropland_2024 = clean_float(cropland_all[year_idx])

population_path = FEATURE_CACHE / 'population_density_2024.npy'
if population_path.exists():
    pop_density_2024 = np.load(population_path, mmap_mode='r')
else:
    with netCDF4.Dataset(HYDE_DIR / 'population_density.nc') as ds:
        dates = netCDF4.num2date(
            ds.variables['time'][:],
            units=ds.variables['time'].units,
            calendar=getattr(ds.variables['time'], 'calendar', 'standard'),
            only_use_cftime_datetimes=False,
            only_use_python_datetimes=False,
        )
        pop_years = np.array([date.year for date in dates])
        pop_idx = int(np.where(pop_years == YEAR)[0][0])
        pop_density_2024 = clean_float(ds.variables['population_density'][pop_idx])
    np.save(population_path, pop_density_2024.astype(np.float32))

def load_cache(name):
    return np.load(require(FEATURE_CACHE / name), mmap_mode='r')

elevation_m = load_cache('elevation_5min.npy')
slope = load_cache('slope_5min.npy')
exclusion = load_cache('exclusion_5min_mode.npy')
rainfed_value = load_cache('rainfed_value_top5_usd_per_ha_checked_36crops.npy')
irrigated_value = load_cache('irrigated_value_top5_usd_per_ha_checked_36crops.npy')
rainfed_calorie = load_cache('rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy')
irrigated_calorie = load_cache('irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy')
city_time = np.load(require(DIST_DIR / 'cities_10_1_12deg_min.npy'), mmap_mode='r')
port_time = np.load(require(DIST_DIR / 'ports_05_1_12deg_min.npy'), mmap_mode='r')
glofas_p10 = np.load(require(GLOFAS_DIR / 'p10_discharge_max_5min_2020.npy'), mmap_mode='r')
distance_river = np.load(require(GLOFAS_DIR / 'distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy'), mmap_mode='r')

print('Loaded grid:', cropland_2024.shape)
print('Finite cropland cells:', int(np.isfinite(cropland_2024).sum()))

In [ ]:
land_mask = (
    np.isfinite(cropland_2024)
    & np.isfinite(pop_density_2024)
    & (pop_density_2024 >= 0)
)
presence = land_mask & (cropland_2024 > PRESENCE_THRESHOLD)
absence = land_mask & ~presence

rng = np.random.default_rng(RANDOM_SEED)
pos_flat = np.flatnonzero(presence.ravel())
zero_flat = np.flatnonzero(absence.ravel())
pos_sample = rng.choice(pos_flat, size=min(N_POS_SAMPLE, len(pos_flat)), replace=False)
zero_sample = rng.choice(zero_flat, size=min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([pos_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, cropland_2024.shape)

def take(array):
    return np.asarray(array[rows, cols])

sample = pd.DataFrame({
    'row': rows.astype(np.int32),
    'col': cols.astype(np.int32),
    'lat': lat[rows].astype(np.float32),
    'lon': lon[cols].astype(np.float32),
    'cropland_fraction': take(cropland_2024).astype(np.float32),
    'presence': (take(cropland_2024) > PRESENCE_THRESHOLD).astype(np.uint8),
    'elevation_m': take(elevation_m).astype(np.float32),
    'slope': take(slope).astype(np.float32),
    'exclusion_class': np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    'log_pop_density_2024': safe_log1p(take(pop_density_2024)),
    'log_city_time_20k_min': safe_log1p(take(city_time)),
    'log_port_time_any_min': safe_log1p(take(port_time)),
    'log_glofas_p10_2020': safe_log1p(take(glofas_p10)),
    'log_distance_river_gt10_2020': safe_log1p(take(distance_river)),
    'log_rainfed_value_top5': safe_log1p(take(rainfed_value)),
    'log_rainfed_calorie_top5': safe_log1p(take(rainfed_calorie)),
    'log_irrigation_value_gain_top5': safe_log1p(np.maximum(take(irrigated_value) - take(rainfed_value), 0)),
    'log_irrigation_calorie_gain_top5': safe_log1p(np.maximum(take(irrigated_calorie) - take(rainfed_calorie), 0)),
})

sample = sample.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
sample['spatial_block'] = (
    np.floor((sample['lat'] + 90) / 10).astype(int) * 36
    + np.floor((sample['lon'] + 180) / 10).astype(int)
)

NUMERIC_COLS = [
    'elevation_m', 'slope', 'log_pop_density_2024',
    'log_city_time_20k_min', 'log_port_time_any_min',
    'log_glofas_p10_2020', 'log_distance_river_gt10_2020',
    'log_rainfed_value_top5', 'log_rainfed_calorie_top5',
    'log_irrigation_value_gain_top5', 'log_irrigation_calorie_gain_top5',
]

positive = sample[sample['presence'].eq(1)].reset_index(drop=True).copy()
print('Sample rows:', len(sample))
print('Positive rows:', len(positive))
print(positive['cropland_fraction'].describe().round(4))

## 2. 相別の明示的なfractional land-use function

第二段階では、農地があるセルだけについて、

```text
E[fraction | positive, X, regime=k] = expit(X beta_k)
```

を推定します。`K=1` は世界共通関数、`K>1` は相別関数です。

この第一版では、各相の専門家関数をfractional-logit quasi-likelihoodで推定し、
EMで潜在相を推定します。テストセルには、学習データから推定したXベースのgateを使います。

In [ ]:
def make_design(train_df, test_df):
    scaler = StandardScaler().fit(train_df[NUMERIC_COLS])
    train_num = scaler.transform(train_df[NUMERIC_COLS])
    test_num = scaler.transform(test_df[NUMERIC_COLS])
    combined = pd.concat([train_df['exclusion_class'], test_df['exclusion_class']], ignore_index=True)
    dummies = pd.get_dummies(combined.astype(str), prefix='exclusion', dtype=float)
    train_dummy = dummies.iloc[:len(train_df)].to_numpy()
    test_dummy = dummies.iloc[len(train_df):].to_numpy()
    X_train = np.column_stack([np.ones(len(train_df)), train_num, train_dummy])
    X_test = np.column_stack([np.ones(len(test_df)), test_num, test_dummy])
    return X_train.astype(float), X_test.astype(float), scaler, list(dummies.columns)


def fit_fractional_expert(X, y, weights=None, init=None):
    # Memory-efficient fractional-logit quasi-likelihood expert.
    # We optimize only the coefficient vector, rather than retaining
    # a full statsmodels GLM result for every EM iteration.
    y = np.clip(np.asarray(y, dtype=float), EPS, 1 - EPS)
    if weights is None:
        weights = np.ones(len(y), dtype=float)
    weights = np.asarray(weights, dtype=float) + EPS
    weight_sum = float(weights.sum())
    if init is None:
        init = np.zeros(X.shape[1], dtype=float)

    def objective(beta):
        eta = np.clip(X @ beta, -35, 35)
        mu = expit(eta)
        qll = y * np.log(mu) + (1 - y) * np.log1p(-mu)
        value = -float(np.sum(weights * qll) / weight_sum)
        value += 0.5 * L2_ALPHA * float(np.sum(beta[1:] ** 2))
        gradient = (X.T @ (weights * (mu - y))) / weight_sum
        gradient[1:] += L2_ALPHA * beta[1:]
        return value, gradient

    result = minimize(
        objective, np.asarray(init, dtype=float), jac=True,
        method='L-BFGS-B', options={'maxiter': 80, 'ftol': 1e-8},
    )
    return {
        'coef': np.asarray(result.x, dtype=float),
        'success': bool(result.success),
        'message': str(result.message),
    }


def predict_expert(model, X):
    return expit(np.clip(X @ model['coef'], -35, 35))


def fit_latent_fractional_logit(X, y, k, random_state=42):
    y = np.clip(np.asarray(y, dtype=float), EPS, 1 - EPS)
    n = len(y)
    if k == 1:
        model = fit_fractional_expert(X, y)
        return {'k': 1, 'models': [model], 'pis': np.array([1.0]), 'responsibilities': np.ones((n, 1)), 'pseudo_loglik': np.nan, 'iterations': 1}

    init_matrix = np.column_stack([y, X[:, 1:]])
    labels = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit_predict(init_matrix)
    responsibilities = np.full((n, k), 0.02 / k, dtype=float)
    responsibilities[np.arange(n), labels] += 0.98
    pis = responsibilities.mean(axis=0)
    previous_ll = -np.inf
    last_ll = -np.inf
    models = []
    previous_coefs = [np.zeros(X.shape[1], dtype=float) for _ in range(k)]

    for iteration in range(1, EM_MAX_ITER + 1):
        models = []
        quasi_loglik = np.empty((n, k), dtype=float)
        for component in range(k):
            model = fit_fractional_expert(
                X, y, responsibilities[:, component], init=previous_coefs[component]
            )
            previous_coefs[component] = model['coef']
            mu = np.clip(predict_expert(model, X), EPS, 1 - EPS)
            quasi_loglik[:, component] = y * np.log(mu) + (1 - y) * np.log(1 - mu)
            models.append(model)

        scores = quasi_loglik + np.log(np.clip(pis, EPS, 1))[None, :]
        normalizer = np.logaddexp.reduce(scores, axis=1)
        new_responsibilities = np.exp(scores - normalizer[:, None])
        new_pis = new_responsibilities.mean(axis=0)
        current_ll = float(np.mean(normalizer))
        last_ll = current_ll

        if abs(current_ll - previous_ll) < EM_TOL * (1 + abs(previous_ll)):
            responsibilities = new_responsibilities
            pis = new_pis
            break
        responsibilities = new_responsibilities
        pis = new_pis
        previous_ll = current_ll

    return {
        'k': k, 'models': models, 'pis': pis,
        'responsibilities': responsibilities,
        'pseudo_loglik': float(last_ll),
        'iterations': iteration,
    }


def fit_x_gate(X, responsibilities, random_state=42):
    k = responsibilities.shape[1]
    if k == 1:
        return None
    hard_labels = responsibilities.argmax(axis=1)
    if len(np.unique(hard_labels)) < 2:
        return None
    gate = LogisticRegression(
        max_iter=300, solver='lbfgs', multi_class='multinomial',
        random_state=random_state,
    )
    gate.fit(X[:, 1:], hard_labels, sample_weight=responsibilities.max(axis=1))
    return gate


def gate_probabilities(gate, mixture, X):
    k = mixture['k']
    if k == 1 or gate is None:
        return np.repeat(mixture['pis'][None, :], len(X), axis=0)
    raw = gate.predict_proba(X[:, 1:])
    out = np.zeros((len(X), k), dtype=float)
    for position, label in enumerate(gate.classes_):
        out[:, int(label)] = raw[:, position]
    return out


def predict_mixture(mixture, weights, X):
    experts = np.column_stack([
        np.clip(predict_expert(model, X), EPS, 1 - EPS)
        for model in mixture['models']
    ])
    return np.sum(weights * experts, axis=1), experts

print('Model functions defined.')

## 3. 空間5分割OOFでK=1,2,3,4を比較する

ここで重要なのは、相を発見した同じデータをそのまま評価に使わないことです。
各foldの学習データだけで相別関数とgateを推定し、未使用の空間ブロックで予測します。

K=1が世界共通のfractional land-use function、K>1が相別モデルです。

In [ ]:
groups = positive['spatial_block'].to_numpy()
y_all = positive['cropland_fraction'].to_numpy(dtype=float)
oof_predictions = {k: np.full(len(positive), np.nan, dtype=float) for k in K_VALUES}
fold_rows = []

gkf = GroupKFold(n_splits=N_SPLITS)
for fold, (train_idx, test_idx) in enumerate(gkf.split(positive, y_all, groups=groups), start=1):
    train_df = positive.iloc[train_idx].copy()
    test_df = positive.iloc[test_idx].copy()
    fit_df = train_df
    if len(fit_df) > MAX_MIXTURE_TRAIN_ROWS:
        fit_df = fit_df.sample(MAX_MIXTURE_TRAIN_ROWS, random_state=RANDOM_SEED + fold)
    fit_positions = fit_df.index.to_numpy()
    train_lookup = pd.Series(np.arange(len(train_df)), index=train_df.index)
    fit_local_idx = train_lookup.loc[fit_positions].to_numpy()

    X_train, X_test, _, _ = make_design(train_df, test_df)
    X_fit = X_train[fit_local_idx]
    y_fit = fit_df['cropland_fraction'].to_numpy(dtype=float)
    y_test = test_df['cropland_fraction'].to_numpy(dtype=float)

    print(f'FOLD {fold}/{N_SPLITS}: train={len(train_df):,}, test={len(test_df):,}, mixture_fit={len(fit_df):,}')
    for k in K_VALUES:
        mixture = fit_latent_fractional_logit(X_fit, y_fit, k=k, random_state=RANDOM_SEED + fold * 100 + k)
        gate = fit_x_gate(X_fit, mixture['responsibilities'], random_state=RANDOM_SEED + fold * 100 + k)
        weights = gate_probabilities(gate, mixture, X_test)
        prediction, _ = predict_mixture(mixture, weights, X_test)
        oof_predictions[k][test_idx] = prediction
        fold_rows.append({
            'fold': fold, 'k': k, 'n_test': len(test_idx),
            'mae': mean_absolute_error(y_test, prediction),
            'rmse': np.sqrt(mean_squared_error(y_test, prediction)),
            'r2': r2_score(y_test, prediction),
            'min_component_share': float(mixture['pis'].min()),
            'max_component_share': float(mixture['pis'].max()),
            'em_iterations': mixture['iterations'],
        })
        del mixture, gate, weights, prediction
        gc.collect()

fold_results = pd.DataFrame(fold_rows)
summary = (
    fold_results.groupby('k', as_index=False)
    .agg(
        mean_mae=('mae', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_r2=('r2', 'mean'),
        sd_r2=('r2', 'std'),
        min_component_share=('min_component_share', 'mean'),
        mean_em_iterations=('em_iterations', 'mean'),
    )
)

print('Spatial OOF comparison')
display(summary.round(4))
fold_results.to_csv(OUT_DIR / 'spatial_oof_fold_results.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUT_DIR / 'spatial_oof_model_comparison.csv', index=False, encoding='utf-8-sig')

In [ ]:
oof_table = positive[['row', 'col', 'lat', 'lon', 'cropland_fraction', 'spatial_block']].copy()
for k in K_VALUES:
    oof_table[f'fraction_pred_k{k}'] = oof_predictions[k]
    oof_table[f'residual_k{k}'] = oof_table['cropland_fraction'] - oof_table[f'fraction_pred_k{k}']

oof_path = OUT_DIR / 'spatial_oof_predictions.csv'
oof_table.to_csv(oof_path, index=False, encoding='utf-8-sig')

# 全データで再推定し、相別の係数と所属確率を保存する。
full_fit = positive
if len(full_fit) > MAX_MIXTURE_TRAIN_ROWS:
    full_fit = full_fit.sample(MAX_MIXTURE_TRAIN_ROWS, random_state=RANDOM_SEED)
X_full, _, _, dummy_columns = make_design(full_fit, full_fit.iloc[:0].copy())
y_full = full_fit['cropland_fraction'].to_numpy(dtype=float)

profile_rows = []
coefficient_rows = []
for k in K_VALUES:
    mixture = fit_latent_fractional_logit(X_full, y_full, k=k, random_state=RANDOM_SEED + k)
    for component, model in enumerate(mixture['models'], start=1):
        params = np.asarray(model['coef'], dtype=float)
        feature_names = ['intercept'] + NUMERIC_COLS + dummy_columns
        for name, value in zip(feature_names, params):
            coefficient_rows.append({
                'k': k, 'component': component, 'feature': name,
                'coefficient_on_logit_fraction': value,
            })
        resp = mixture['responsibilities'][:, component - 1]
        profile_rows.append({
            'k': k, 'component': component,
            'mixture_weight': mixture['pis'][component - 1],
            'mean_posterior_membership': resp.mean(),
            'mean_observed_fraction_weighted': np.average(y_full, weights=resp),
            'pseudo_loglik': mixture['pseudo_loglik'],
            'em_iterations': mixture['iterations'],
        })

profiles = pd.DataFrame(profile_rows)
coefficients = pd.DataFrame(coefficient_rows)
profiles.to_csv(OUT_DIR / 'full_sample_component_profiles.csv', index=False, encoding='utf-8-sig')
coefficients.to_csv(OUT_DIR / 'full_sample_component_coefficients.csv', index=False, encoding='utf-8-sig')

print('Saved:', oof_path)
print('Component profiles')
display(profiles.round(4))
print('Coefficient table:', OUT_DIR / 'full_sample_component_coefficients.csv')

## 読み方と次の段階

- `K=1`：世界共通の明示的なfractional land-use function。
- `K>1`：相別の明示的な関数。各相の係数と、セルごとの潜在所属確率が得られる。
- 相の数は、OOFのRMSE/R²だけでなく、相のサイズ、係数の安定性、初期値を変えたときの再現性で判断する。
- 現段階では第二段階だけを相別化している。第一段階のpresenceも同じ相で説明するのは、第二段階で相の存在が確認できてから行う。
- 現在の特徴量は自然・水・市場・地形の代理変数であり、労働資本、農業資本、制度資本を直接測っているわけではない。そこは次のデータ拡張で追加する。

この第一版で相別モデルが安定して改善するなら、次は

```text
同じ latent regime z
    -> cropland presence function
    -> positive cropland fraction function
    -> crop-choice function
    -> yield-attainment function
```

という共有潜在相モデルへ拡張する。